In [9]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from scripts.Loading_Dataset import LiverDataset
from tqdm import tqdm

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [11]:
#Data augmentation and normalization for encoding
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [12]:
#Load dataset
root_dir = "./Dataset" 

dataset = LiverDataset(root_dir=root_dir, transform=transform)
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [13]:
#Load Resnet 18 Encoder (pretrained on ImageNet)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# REMOVE CLASSIFIER HEAD
model.fc = nn.Identity()

model = model.to(device)
model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [14]:
all_features = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(dataloader, desc="Extracting features"):
        images = images.to(device)

        # forward pass → 512-d embeddings
        features = model(images)

        # flatten just in case (already [B, 512])
        features = features.view(features.size(0), -1)

        all_features.append(features.cpu())
        all_labels.append(labels)

# combine all batches
all_features = torch.cat(all_features, dim=0)
all_labels = torch.cat(all_labels, dim=0)

Extracting features:   0%|          | 0/112 [00:00<?, ?it/s]

Extracting features: 100%|██████████| 112/112 [08:16<00:00,  4.43s/it] 


In [15]:
#Output
print("Feature shape:", all_features.shape)
print("Label shape:", all_labels.shape)

print("\nSample feature vector:")
print(all_features[0])


Feature shape: torch.Size([3557, 512])
Label shape: torch.Size([3557])

Sample feature vector:
tensor([1.9493e+00, 1.5969e+00, 2.5116e+00, 1.5285e+00, 4.9267e-01, 2.6831e-01,
        3.4734e+00, 0.0000e+00, 4.4180e-01, 1.7552e+00, 9.7500e-01, 3.0314e+00,
        5.3093e-01, 8.3093e-01, 1.7340e+00, 6.3898e-01, 8.1549e-02, 1.1279e+00,
        1.0849e+00, 1.2908e-01, 7.2132e-02, 2.4732e-01, 8.5629e-01, 1.0142e+00,
        1.4797e+00, 1.1192e+00, 1.6240e+00, 2.0191e+00, 2.5668e-03, 3.0239e-01,
        6.0701e-01, 5.6664e-01, 7.4610e-01, 1.9786e+00, 4.3061e-01, 0.0000e+00,
        5.8350e-03, 2.9929e+00, 8.7246e-01, 1.1260e+00, 2.6761e-03, 1.9127e+00,
        3.8899e-01, 7.6898e-01, 1.6494e-01, 3.1124e+00, 6.9117e-01, 2.8230e-02,
        4.7518e-01, 1.2274e+00, 0.0000e+00, 3.5473e-01, 7.6281e-01, 2.6620e-01,
        1.3074e+00, 2.3102e-01, 3.3768e-01, 1.3345e+00, 7.1803e-01, 3.2066e-01,
        2.1426e-01, 0.0000e+00, 3.7340e-01, 2.4003e+00, 3.1938e-01, 3.5095e-01,
        8.9867e-01, 1.433